In [ ]:
"""
03_dimensionality_reduction.ipynb
==================================
Applies feature selection and linear transformation to preprocessed profiles.
Follows Step 2 of the analysis pipeline.

Inputs:
    - data/processed/all_profiles.csv for now  (later, preprocessed output from 02_preprocess.ipynb)

Outputs:
    - TBD
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA


DATA_DIR = Path("../data/processed")
df = pd.read_csv(DATA_DIR / "all_profiles.csv")

meta_cols = ["plate_name", "session", "Row", "Column", "compound", "dose_uM", "cell_type", "condition"]
feature_cols = [c for c in df.columns if c not in meta_cols]

In [ ]:
"""
NOTE: the following function placeholders are just rough ideas corresponding to the steps. 
Please feel free to adjust or add anything, I can integrate later :)
"""

In [ ]:
# Ideas for helper functions for 2a. Feature Selection

def variance_threshold(df, feature_cols, threshold=1e-6):
    """
    Removes near-zero variance features (Step 2a).
    Fast first pass before more expensive filtering methods.
    Report how many features are removed.
    """
    selector = VarianceThreshold(threshold=threshold)
    selector.fit(df[feature_cols])

    retained = [c for c, keep in zip(feature_cols, selector.get_support()) if keep]
    dropped  = [c for c, keep in zip(feature_cols, selector.get_support()) if not keep]

    print(f"[variance_threshold] Dropped {len(dropped)} features with variance < {threshold}")
    print(f"[variance_threshold] Retained {len(retained)} features")

    return retained


def correlation_filter(df, feature_cols, threshold=0.95):
    """
    Removes redundant features via iterative correlation filtering (Step 2a).
    At each step, removes the feature with the highest mean absolute correlation
    to all others when any pair exceeds the threshold.
    Report how many features remain after filtering.
    """
    cols = list(feature_cols)
    dropped = []

    while True:
        corr = df[cols].corr(method="pearson").abs()
        np.fill_diagonal(corr.values, 0)  # ignore self-correlation

        over = corr >= threshold
        if not over.any().any():
            break

        # Drop the feature with the highest mean absolute correlation to all others
        mean_corr = corr.mean(axis=1)
        worst = mean_corr[over.any(axis=1)].idxmax()
        cols.remove(worst)
        dropped.append(worst)

    print(f"[correlation_filter] Dropped {len(dropped)} features with |r| >= {threshold}")
    print(f"[correlation_filter] Retained {len(cols)} features")

    return cols


def replicate_correlation_filter(df, feature_cols):
    """
    Removes features that are inconsistent across biological replicates (Step 2a).
    Uses NEG CTRL wells as replicates — healthy wells and disease wells should each
    be morphologically identical within a plate.
    Work at the plate level to avoid cross-plate batch effects.
    Compare retained feature set to correlation filtering alone.

    Args:
        df: profiles dataframe
        feature_cols: list of feature column names

    Returns:
        filtered list of feature column names
    """
    pass


def cohen_d_filter(df, feature_cols):
    """
    Selects features that discriminate healthy from disease using Cohen's d (Step 2a).
    Recompute per session since the healthy/disease separation may drift across runs.
    Suggested threshold methods: Otsu (recommended), GMM, permutation null, or elbow.

    Args:
        df: profiles dataframe
        feature_cols: list of feature column names

    Returns:
        filtered list of feature column names
    """
    pass

In [4]:
# Ideas for helper functions for 2b. Linear Transformation

def fit_pca(df, feature_cols, plot=False):
    # Fit on controls only
    controls = df[df["condition"].isin(["healthy", "disease"])]
    control_features = controls[feature_cols].values
    
    # Fit PCA
    pca = PCA()
    pca.fit(control_features)
    
    # Determine number of PCs for ~90% variance
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n_pcs = np.argmax(cumvar >= 0.9) + 1
    
    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(pca.explained_variance_ratio_[:n_pcs + 5], marker='o', markersize=3)
        axes[0].set_xlabel("PC")
        axes[0].set_ylabel("Variance Explained (fraction)")
        axes[0].set_title("Scree Plot")
        
        axes[1].plot(cumvar[:n_pcs + 5], marker='o', markersize=3)
        axes[1].axhline(y=0.9, color='r', linestyle='--', label='90% threshold')
        axes[1].set_xlabel("PC")
        axes[1].set_ylabel("Cumulative Variance Explained")
        axes[1].legend()
        plt.tight_layout()
        plt.show()
    
    # Project conditions using fitted PCA
    all_features = df[feature_cols].values
    scores = pca.transform(all_features)[:, :n_pcs]
    
    # Add PC columns to df
    pc_cols = [f"PC{i+1}" for i in range(n_pcs)]
    df_out = df.copy()
    for i, col in enumerate(pc_cols):
        df_out[col] = scores[:, i]
    
    return df_out, pca


def apply_sphering(df, pca_cols):
    """
    Applies ZCA whitening (sphering) fitted on healthy wells only (Step 2b).
    Transforms the PCA space so features have equal variance and are uncorrelated,
    with the healthy state defining the geometry.
    Confirm healthy wells form a roughly spherical cloud in the first 2 PCs after whitening.

    Args:
        df: profiles dataframe with PCA scores
        pca_cols: list of PC column names

    Returns:
        df with sphered PCA scores
    """
    # Fit whitening on healthy wells only
    healthy_mask = df["condition"] == "healthy"
    healthy_scores = df.loc[healthy_mask, pca_cols].values
    
    # Compute healthy mean and std per PC
    healthy_mean = healthy_scores.mean(axis=0)
    healthy_std = healthy_scores.std(axis=0)
    
    # Apply to all wells: center on healthy mean, scale by healthy std
    df_out = df.copy()
    for i, col in enumerate(pca_cols):
        df_out[col] = (df[col] - healthy_mean[i]) / healthy_std[i]
    
    return df_out

In [ ]:
# Ideas for overall pipeline
# feel free to reorganize, swap out methods, or add steps as needed
# note: cohen_d_filter and fit_pca should be applied per session (see Step 2c in workplan)

feature_cols = variance_threshold(df, feature_cols)
feature_cols = correlation_filter(df, feature_cols)
feature_cols = replicate_correlation_filter(df, feature_cols)
feature_cols = cohen_d_filter(df, feature_cols)
df, pca = fit_pca(df, feature_cols)
df = apply_sphering(df, pca_cols=[c for c in df.columns if c.startswith("PC")])

In [ ]:
# Step 2c: per-session label-dependent selection + PCA

results = []
for session, session_df in df.groupby("session"):
    session_feature_cols = cohen_d_filter(session_df, feature_cols)
    session_df, pca = fit_pca(session_df, session_feature_cols)
    session_df = apply_sphering(session_df, pca_cols=[c for c in session_df.columns if c.startswith("PC")])
    results.append(session_df)

df = pd.concat(results, ignore_index=True)

# TODO: repeat batch detection + correction from 02_preprocess on PCA-summarized data